# 02. 노드 분류 — GCN으로 Cora 풀기

**Cora** 는 GNN의 대표 벤치마크입니다. 노드는 논문, 엣지는 인용 관계이며,
각 논문을 7개 주제 중 하나로 분류합니다. 일부 노드만 라벨이 있는 **준지도 학습** 문제입니다.

2층 GCN으로 노드 분류를 수행합니다.

1. Cora 데이터 로드
2. GCN 모델 정의
3. 학습 및 평가

> Cora 데이터셋은 노트북 실행 시 자동 다운로드됩니다.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Cora 데이터 로드

Cora는 노드 2708개, 7개 클래스, 노드당 1433차원 특징을 가집니다.
`train_mask`/`val_mask`/`test_mask`로 학습·검증·평가 노드를 구분합니다.

In [ ]:
dataset = Planetoid(root='/workspace/data/Cora', name='Cora')
data = dataset[0].to(device)

print('클래스 수:', dataset.num_classes)
print('노드 특징 차원:', dataset.num_node_features)
print('노드 수:', data.num_nodes, '| 엣지 수:', data.num_edges)
print('학습/검증/평가 노드:', int(data.train_mask.sum()),
      int(data.val_mask.sum()), int(data.test_mask.sum()))

## 2. GCN 모델

2층 GCN을 정의합니다. 첫 층은 특징을 16차원 은닉 표현으로, 둘째 층은 클래스 수만큼으로 매핑합니다.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)

model = GCN(dataset.num_node_features, 16, dataset.num_classes).to(device)
print(model)

## 3. 학습 및 평가

학습 노드(140개)에 대해서만 손실을 계산하지만, 그래프 전체 구조를 활용해 학습합니다.
학습 전후 정확도를 비교합니다.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def accuracy(mask):
    model.eval()
    with torch.no_grad():
        pred = model(data.x, data.edge_index).argmax(1)
        return int((pred[mask] == data.y[mask]).sum()) / int(mask.sum())

acc_before = accuracy(data.test_mask)
print(f'학습 전 테스트 정확도: {acc_before:.3f}')

for epoch in range(200):
    model.train(); optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); optimizer.step()

acc_after = accuracy(data.test_mask)
print(f'학습 후 테스트 정확도: {acc_after:.3f}')

### 정리

- 2층 GCN으로 Cora 노드를 분류했습니다. 학습 전 무작위 수준에서 학습 후 80% 안팎까지 향상됩니다.
- GNN은 노드 자신의 특징뿐 아니라 **이웃(인용 관계)** 정보를 활용해 분류 성능을 높입니다.
- GCNConv를 GATConv·SAGEConv로 바꾸면 다른 GNN 구조를 실험할 수 있습니다.
- 다음 노트북에서는 노드가 아닌 **그래프 전체**를 분류합니다.